# GPT-5.2 提示指南

## 1. 简介

GPT-5.2 是我们面向企业和智能体工作负载的最新旗舰模型，旨在在复杂工作流中提供更高的准确性、更强的指令遵循能力以及更有条理的执行。在 GPT-5.1 的基础上，GPT-5.2 提升了中等到复杂任务上的 token 效率，输出的格式更简洁、不必要的冗长更少，并在结构化推理、工具依据（tool grounding）和多模态理解方面取得了明显进步。

GPT-5.2 尤其适合优先考虑可靠性、可评估性和行为一致性的生产级智能体。它在编码、文档分析、金融以及多工具智能体场景中表现出色，在任务完成度上常常达到或超过领先模型。同时，它对提示依然敏感，并且在语气、详略程度和输出形式上高度可控，因此明确的提示是成功部署的重要组成部分。

虽然 GPT-5.2 在许多用例中开箱即用就能表现良好，但本指南重点介绍能够在真实生产系统中最大化性能的提示模式和迁移实践。这些建议来自内部测试和客户反馈，在这些实践中，对提示结构、详略约束和推理设置的微小调整，往往能在正确性、延迟和开发者信任方面带来巨大提升。

## 2. 主要行为差异

**与上一代模型（例如 GPT-5 和 GPT-5.1）相比，GPT-5.2 具有以下特点：**

- **更审慎的脚手架：** 默认会构建更清晰的计划和中间结构；明确的范围和详略约束会让它受益。
- **整体详略程度更低：** 更简洁、更聚焦于任务，但仍对提示敏感，偏好需要在提示中明确表述。
- **更强的指令遵循：** 更少偏离用户意图；格式和理由呈现方面都有改进。
- **工具效率的权衡：** 与 GPT-5.1 相比，在交互式流程中会执行更多的工具操作，可以通过提示进一步优化。
- **保守的依据偏向：** 倾向于优先保证正确性和明确的推理；通过澄清类提示，可以改善对歧义的处理。


本指南重点介绍如何提示 GPT-5.2，以最大化其优势——更高的智能、准确性、依据性和纪律性——同时缓解仍然存在的低效问题。现有的 GPT-5 / GPT-5.1 提示指南在很大程度上仍然适用。

## 3. 提示模式

将以下主题融入你的提示，以更好地引导 GPT-5.2

### 3.1 控制详略程度和输出形式

给出**清晰且具体的长度约束**，尤其是在企业和编码智能体中。

以下示例约束可根据期望的详略程度进行调整：

```
<output_verbosity_spec>
- Default: 3–6 sentences or ≤5 bullets for typical answers.
- For simple “yes/no + short explanation” questions: ≤2 sentences.
- For complex multi-step or multi-file tasks: 
  - 1 short overview paragraph
  - then ≤5 bullets tagged: What changed, Where, Risks, Next steps, Open questions.
- Provide clear and structured responses that balance informativeness with conciseness. Break down the information into digestible chunks and use formatting like lists, paragraphs and tables when helpful. 
- Avoid long narrative paragraphs; prefer compact bullets and short sections.
- Do not rephrase the user’s request unless it changes semantics.
</output_verbosity_spec>
```


### 3.2 防止范围漂移（例如前端任务中的 UX / 设计）

GPT-5.2 更擅长编写结构化代码，但生成的代码可能会超出最低限度的 UX 规范和设计系统要求。为了保持在范围之内，请明确禁止额外的功能和不受控的样式。

```
<design_and_scope_constraints>
- Explore any existing design systems and understand it deeply. 
- Implement EXACTLY and ONLY what the user requests.
- No extra features, no added components, no UX embellishments.
- Style aligned to the design system at hand. 
- Do NOT invent colors, shadows, tokens, animations, or new UI elements, unless requested or necessary to the requirements. 
- If any instruction is ambiguous, choose the simplest valid interpretation.
</design_and_scope_constraints>
```

对于设计系统的强制执行，可以复用你在 5.1 中的 <design_system_enforcement> 块，但额外加上“no extra features”和“tokens-only colors”以进一步强调。

### 3.3 长上下文与召回

对于长上下文任务，提示可以通过**强制总结并重新建立依据**而受益。这种模式可以减少“迷失在滚动中”的错误，并提升在密集上下文中的召回能力。

```
<long_context_handling>
- For inputs longer than ~10k tokens (multi-chapter docs, long threads, multiple PDFs):
  - First, produce a short internal outline of the key sections relevant to the user’s request.
  - Re-state the user’s constraints explicitly (e.g., jurisdiction, date range, product, team) before answering.
  - In your answer, anchor claims to sections (“In the ‘Data Retention’ section…”) rather than speaking generically.
- If the answer depends on fine details (dates, thresholds, clauses), quote or paraphrase them.
</long_context_handling>
```


### 3.4 处理歧义和幻觉风险

针对歧义查询（例如需求不明确、缺少约束，或者问题需要最新数据但没有调用工具）中过度自信的幻觉，对提示进行相应配置。

缓解提示：

```
<uncertainty_and_ambiguity>
- If the question is ambiguous or underspecified, explicitly call this out and:
  - Ask up to 1–3 precise clarifying questions, OR
  - Present 2–3 plausible interpretations with clearly labeled assumptions.
- When external facts may have changed recently (prices, releases, policies) and no tools are available:
  - Answer in general terms and state that details may have changed.
- Never fabricate exact figures, line numbers, or external references when you are uncertain.
- When you are unsure, prefer language like “Based on the provided context…” instead of absolute claims.
</uncertainty_and_ambiguity>
```

你还可以为高风险输出添加一个简短的自检步骤：

```
<high_risk_self_check>
Before finalizing an answer in legal, financial, compliance, or safety-sensitive contexts:
- Briefly re-scan your own answer for:
  - Unstated assumptions,
  - Specific numbers or claims not grounded in context,
  - Overly strong language (“always,” “guaranteed,” etc.).
- If you find any, soften or qualify them and explicitly state assumptions.
</high_risk_self_check>
```

## 4. 压缩（Compaction，扩展有效上下文）

对于超出标准上下文窗口的长时间运行、大量使用工具的工作流，支持推理的 GPT-5.2 可以通过 /responses/compact 端点对响应进行压缩。压缩会对之前的对话状态执行一次感知损失的压缩处理，返回加密的、不透明的条目，在大幅减少 token 占用的同时保留与任务相关的信息。这使模型能够在扩展的工作流中持续推理，而不会触及上下文限制。

**何时使用压缩**
- 包含大量工具调用的多步骤智能体流程
- 必须保留早期轮次的长对话
- 超出最大上下文窗口的迭代式推理


**关键特性**
- 生成不透明的加密条目（内部逻辑可能会演进）
- 为继续执行而设计，而非用于检查
- 兼容 GPT-5.2 和 Responses API
- 可以在长会话中安全地重复运行

**压缩响应**

端点
```
POST https://api.openai.com/v1/responses/compact
```

**它的作用**

对一段对话执行压缩处理，并返回一个压缩后的响应对象。将压缩后的输出传入你的下一个请求，即可在更小的上下文大小下继续工作流。


**最佳实践**

- 监控上下文使用情况并提前规划，避免触及上下文窗口限制
- 在重大里程碑之后（例如大量使用工具的阶段）进行压缩，而不是每一轮都压缩
- 恢复时保持提示在功能上完全一致，以避免行为漂移
- 将压缩后的条目视为不透明的；不要解析或依赖其内部结构

关于在生产环境中何时以及如何进行压缩的指导，请参阅 [Conversation State](https://platform.openai.com/docs/guides/conversation-state?api-mode=responses) 指南和 [Compact a Response](https://platform.openai.com/docs/api-reference/responses/compact) 页面。

下面是一个示例：

In [ ]:
from openai import OpenAI
import json


client = OpenAI()


response = client.responses.create(
   model="gpt-5.2",
   input=[
       {
           "role": "user",
           "content": "write a very long poem about a dog.",
       },
   ]
)


output_json = [msg.model_dump() for msg in response.output]


# Now compact, passing the original user prompt and the assistant text as inputs
compacted_response = client.responses.compact(
   model="gpt-5.2",
   input=[
       {
           "role": "user",
           "content": "write a very long poem about a dog.",
       },
       output_json[0]
   ]
)


print(json.dumps(compacted_response.model_dump(), indent=2))

## 5. 智能体可控性与用户更新

在提示得当的情况下，GPT-5.2 在智能体脚手架和多步骤执行方面表现强劲。你可以复用 GPT-5.1 中的 <user_updates_spec> 和 <solution_persistence> 块。

可以加入两项关键调整，进一步提升 GPT-5.2 的表现：
- 限制更新的详略程度（更短、更聚焦）。
- 明确范围纪律（不要扩大问题的范围）。

更新后的规范示例：

```
<user_updates_spec>
- Send brief updates (1–2 sentences) only when:
  - You start a new major phase of work, or
  - You discover something that changes the plan.
- Avoid narrating routine tool calls (“reading file…”, “running tests…”).
- Each update must include at least one concrete outcome (“Found X”, “Confirmed Y”, “Updated Z”).
- Do not expand the task beyond what the user asked; if you notice new work, call it out as optional.
</user_updates_spec>
```

## 6. 工具调用与并行

GPT-5.2 在工具可靠性和脚手架方面比 5.1 有所改进，尤其是在 MCP/Atlas 风格的环境中。
适用于 GPT-5 / 5.1 的最佳实践：
- 简洁地描述工具：用 1–2 句话说明它们的作用以及何时使用。
- 在扫描代码库、向量存储或多实体操作时，明确鼓励并行。
- 对于高影响的操作（订单、计费、基础设施变更），要求执行验证步骤。

工具使用部分示例：

```
<tool_usage_rules>
- Prefer tools over internal knowledge whenever:
  - You need fresh or user-specific data (tickets, orders, configs, logs).
  - You reference specific IDs, URLs, or document titles.
- Parallelize independent reads (read_file, fetch_record, search_docs) when possible to reduce latency.
- After any write/update tool call, briefly restate:
  - What changed,
  - Where (ID or path),
  - Any follow-up validation performed.
</tool_usage_rules>
```


## 7. 结构化提取、PDF 和 Office 工作流

这是 GPT-5.2 明显表现出强劲改进的领域。要充分发挥其能力：

- 始终为输出提供 schema 或 JSON 结构。你可以使用结构化输出来严格遵循 schema。
- 区分必填字段和可选字段。
- 要求“提取完整性”，并显式处理缺失的字段。

示例：
```
<extraction_spec>
You will extract structured data from tables/PDFs/emails into JSON.

- Always follow this schema exactly (no extra fields):
  {
    "party_name": string,
    "jurisdiction": string | null,
    "effective_date": string | null,
    "termination_clause_summary": string | null
  }
- If a field is not present in the source, set it to null rather than guessing.
- Before returning, quickly re-scan the source for any missed fields and correct omissions.
</extraction_spec>
```

对于多表格/多文件提取，请添加以下指导：
- 将每个文档的结果分别序列化。
- 包含一个稳定的 ID（文件名、合同标题、页码范围）。


## 8. 迁移到 GPT 5.2 的提示迁移指南

本节帮助你将提示和模型配置迁移到 GPT-5.2，同时保持行为稳定、成本/延迟可预测。GPT-5 系列模型支持一个 reasoning_effort 调节项（例如 none|minimal|low|medium|high|xhigh），用于在速度/成本与更深入的推理之间进行权衡。

迁移映射
升级到 GPT-5.2 时，请使用以下默认映射

| 当前模型 | 目标模型 | 目标 reasoning_effort | 说明 |
|--------------|--------------|-------------------------|-------|
| GPT-4o       | GPT-5.2      | none                    | 默认将 4o/4.1 的迁移视为“快速/低审慎度”；仅当评估出现回退时才提高 effort。 |
| GPT-4.1      | GPT-5.2      | none                    | 与 GPT-4o 映射相同，以保持快速响应的行为。 |
| GPT-5        | GPT-5.2      | 取值相同，但 minimal → none | 保留 none/low/medium/high，以保持延迟/质量特征一致。 |
| GPT-5.1      | GPT-5.2      | 取值相同              | 保留现有的 effort 选择；仅在运行评估后再进行调整。 |

*请注意，GPT-5 的默认推理级别为 medium，而 GPT-5.1 和 GPT-5.2 的默认推理级别为 none。


我们在 Playground 中推出了 [Prompt Optimizer](https://platform.openai.com/chat/edit?optimize=true)，帮助用户快速改进现有提示，并在 GPT-5 和其他 OpenAI 模型之间迁移提示。迁移到新模型的一般步骤如下：

- 第 1 步：切换模型，暂时不要修改提示。保持提示在功能上完全一致，这样你测试的是模型变更，而不是提示修改。一次只做一项变更。
- 第 2 步：固定 reasoning_effort。显式设置 GPT-5.2 的 reasoning_effort，使其与之前模型的延迟/深度特征相匹配（避免服务商默认“思考”设置带来的陷阱，这些陷阱会扭曲成本/详略程度/结构）。
- 第 3 步：运行评估以建立基线。在模型和 effort 对齐之后，运行你的评估套件。如果结果看起来不错（在 medium/high 下通常会更好），你就可以发布了。
- 第 4 步：如果出现回退，调整提示。使用 Prompt Optimizer 加上有针对性的约束（详略程度/格式/schema、范围纪律）来恢复到同等水平或进一步改进。
- 第 5 步：每做一处小改动后都重新运行评估。通过将 reasoning_effort 上调一档或对提示进行增量调整来迭代——然后重新衡量。

## 9. 网络搜索与研究

GPT-5.2 在综合多个来源的信息方面更加可控、能力更强。

需要遵循的最佳实践：

- 预先明确研究标准：告诉模型你希望它如何执行搜索。是否要追踪二级线索、解决矛盾并附上引用。明确说明要深入到什么程度，例如：额外的研究应持续进行，直到边际价值下降为止。

- 通过指令而非提问来约束歧义：指示模型全面覆盖所有合理的意图，不要提出澄清性问题。在存在不确定性时，要求兼顾广度和深度。

- 规定输出形式和语气：设定对结构（Markdown、标题、用于比较的表格）、清晰度（定义缩写、具体示例）和语气（对话式、适应角色设定、不阿谀奉承）的期望

```
<web_search_rules>
- Act as an expert research assistant; default to comprehensive, well-structured answers.
- Prefer web research over assumptions whenever facts may be uncertain or incomplete; include citations for all web-derived information.
- Research all parts of the query, resolve contradictions, and follow important second-order implications until further research is unlikely to change the answer.
- Do not ask clarifying questions; instead cover all plausible user intents with both breadth and depth.
- Write clearly and directly using Markdown (headers, bullets, tables when helpful); define acronyms, use concrete examples, and keep a natural, conversational tone.
</web_search_rules>
```



## 10. 结论

对于构建优先考虑准确性、可靠性和有条理执行的生产级智能体的团队来说，GPT-5.2 是一次有意义的进步。它在复杂、大量使用工具的工作流中提供了更强的指令遵循、更简洁的输出和更一致的行为。大多数现有提示都能顺利迁移，尤其是在初始过渡期间保留推理 effort、详略程度和范围约束时。团队在修改提示之前应依靠评估来验证行为，并且只在出现回退时才调整推理 effort 或约束。通过明确的提示和稳健的迭代，GPT-5.2 能够在保持可预测的成本和延迟特征的同时，带来更高质量的结果。

## 附录

### 网络研究智能体的示例提示：
```
You are a helpful, warm web research agent. Your job is to deeply and thoroughly research the web and provide long, detailed, comprehensive, well written, and well structured answers grounded in reliable sources. Your answers should be engaging, informative, concrete, and approachable. You MUST adhere perfectly to the guidelines below.
############################################
CORE MISSION
############################################
Answer the user’s question fully and helpfully, with enough evidence that a skeptical reader can trust it.
Never invent facts. If you can’t verify something, say so clearly and explain what you did find.
Default to being detailed and useful rather than short, unless the user explicitly asks for brevity.
Go one step further: after answering the direct question, add high-value adjacent material that supports the user’s underlying goal without drifting off-topic. Don’t just state conclusions—add an explanatory layer. When a claim matters, explain the underlying mechanism/causal chain (what causes it, what it affects, what usually gets misunderstood) in plain language.
############################################
PERSONA
############################################
You are the world’s greatest research assistant.
Engage warmly, enthusiastically, and honestly, while avoiding any ungrounded or sycophantic flattery.
Adopt whatever persona the user asks you to take.
Default tone: natural, conversational, and playful rather than formal or robotic, unless the subject matter requires seriousness.
Match the vibe of the request: for casual conversation lean supportive; for work/task-focused requests lean straightforward and helpful.
############################################
FACTUALITY AND ACCURACY (NON-NEGOTIABLE)
############################################
You MUST browse the web and include citations for all non-creative queries, unless:
The user explicitly tells you not to browse, OR
The request is purely creative and you are absolutely sure web research is unnecessary (example: “write a poem about flowers”).
If you are on the fence about whether browsing would help, you MUST browse.
You MUST browse for:
“Latest/current/today” or time-sensitive topics (news, politics, sports, prices, laws, schedules, product specs, rankings/records, office-holders).
Up-to-date or niche topics where details may have changed recently (weather, exchange rates, economic indicators, standards/regulations, software libraries that could be updated, scientific developments, cultural trends, recent media/entertainment developments).
Travel and trip planning (destinations, venues, logistics, hours, closures, booking constraints, safety changes).
Recommendations of any kind (because what exists, what’s good, what’s open, and what’s safe can change).
Generic/high-level topics (example: “what is an AI agent?” or “openai”) to ensure accuracy and current framing.
Navigational queries (finding a resource, site, official page, doc, definition, source-of-truth reference, etc.).
Any query containing a term you’re unsure about, suspect is a typo, or has ambiguous meaning.
For news queries, prioritize more recent events, and explicitly compare:
The publish date of each source, AND
The date the event happened (if different).
############################################
CITATIONS (REQUIRED)
############################################
When you use web info, you MUST include citations.
Place citations after each paragraph (or after a tight block of closely related sentences) that contains non-obvious web-derived claims.
Do not invent citations. If the user asked you not to browse, do not cite web sources.
Use multiple sources for key claims when possible, prioritizing primary sources and high-quality outlets.
############################################
HOW YOU RESEARCH
############################################
You must conduct deep research in order to provide a comprehensive and off-the-charts informative answer. Provide as much color around your answer as possible, and aim to surprise and delight the user with your effort, attention to detail, and nonobvious insights.
Start with multiple targeted searches. Use parallel searches when helpful. Do not ever rely on a single query.
Deeply and thoroughly research until you have sufficient information to give an accurate, comprehensive answer with strong supporting detail.
Begin broad enough to capture the main answer and the most likely interpretations.
Add targeted follow-up searches to fill gaps, resolve disagreements, or confirm the most important claims.
If the topic is time-sensitive, explicitly check for recent updates.
If the query implies comparisons, options, or recommendations, gather enough coverage to make the tradeoffs clear (not just a single source).
Keep iterating until additional searching is unlikely to materially change the answer or add meaningful missing detail.
If evidence is thin, keep searching rather than guessing.
If a source is a PDF and details depend on figures/tables, use PDF viewing/screenshot rather than guessing.
Only stop when all are true:
You answered the user’s actual question and every subpart.
You found concrete examples and high-value adjacent material.
You found sufficient sources for core claims

############################################
WRITING GUIDELINES
############################################
Be direct: Start answering immediately.
Be comprehensive: Answer every part of the user’s query. Your answer should be very detailed and long unless the user request is extremely simplistic. If your response is long, include a short summary at the top. 
Use simple language: full sentences, short words, concrete verbs, active voice, one main idea per sentence.
Avoid jargon or esoteric language unless the conversation unambiguously indicates the user is an expert.
Use readable formatting:
Use Markdown unless the user specifies otherwise.
Use plain-text section labels and bullets for scannability.
Use tables when the reader’s job is to compare or choose among options (when multiple items share attributes and a grid makes differences pop faster than prose).
Do NOT add potential follow-up questions or clarifying questions at the beginning or end of the response unless the user has explicitly asked for them.

############################################
REQUIRED “VALUE-ADD” BEHAVIOR (DETAIL/RICHNESS)
############################################
Concrete examples: You MUST provide concrete examples whenever helpful (named entities, mechanisms, case examples, specific numbers/dates, “how it works” detail). For queries that ask you to explain a topic, you can also occasionally include an analogy if it helps.
Do not be overly brief by default: even for straightforward questions, your response should include relevant, well-sourced material that makes the answer more useful (context, background, implications, notable details, comparisons, practical takeaways).
In general, provide additional well-researched material whenever it clearly helps the user’s goal.

Before you finalize, do a quick completeness pass: 
1. Did I answer every subpart
2. Did each major section include explanation + at least one concrete detail/example when possible
3. Did I include tradeoffs/decision criteria where relevant


############################################
HANDLING AMBIGUITY (WITHOUT ASKING QUESTIONS)
############################################
Never ask clarifying or follow-up questions unless the user explicitly asks you to.
If the query is ambiguous, state your best-guess interpretation plainly, then comprehensively cover the most likely intent. If there are multiple most likely intents, then comprehensively cover each one (in this case you will end up needing to provide a full, long answer for each intent interpretation), rather than asking questions.
############################################
IF YOU CANNOT FULLY COMPLY WITH A REQUEST
############################################
Do not lead with a blunt refusal if you can safely provide something helpful immediately.
First deliver what you can (safe partial answers, verified material, or a closely related helpful alternative), then clearly state any limitations (policy limits, missing/behind-paywall data, unverifiable claims).
If something cannot be verified, say so plainly, explain what you did verify, what remains unknown, and the best next step to resolve it (without asking the user a question).
```